In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, cross_validate, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# Cargar datos completos
with open("data_dict_complete.pkl", "rb") as f:
    data_dict = pickle.load(f)

# Cargar mejores hiperparámetros
with open("best_hyperparams_all_models.pkl", "rb") as f:
    best_hyperparams = pickle.load(f)

# Definir longitudes de onda (asumiendo 501 puntos de 1100 a 2100 nm)
wavelengths = np.linspace(1100, 2100, 501)

# Definir los rangos espectrales a evaluar
rangos = {
    'completo': None,          # todas las variables
    'hasta_1700': 1700,
    'hasta_1400': 1400,
}

# Configuración de validación cruzada
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Diccionario para almacenar todos los resultados por combinación
# Estructura: results_cv[modelo][pretratamiento][rango] = {'rmse': [], 'r2': [], 'mae': [], 'rpd': []}
results_cv = {
    'svr': {},
    'random_forest': {},
    'cnn': {}
}

## Funcion para obtener datos reducidos

In [2]:
def get_data_by_range(data_dict, pretratamiento, rango_limite):
    """
    Retorna X_all, y_all para un pretratamiento dado, con reducción de columnas si se especifica rango.
    rango_limite = None -> todas las variables
    rango_limite = 1700 -> hasta 1700 nm
    rango_limite = 1400 -> hasta 1400 nm
    """
    X_all = data_dict[pretratamiento]['X']
    y_all = data_dict[pretratamiento]['Y'].ravel()
    
    if rango_limite is not None:
        # Calcular índice máximo
        idx_max = np.where(wavelengths <= rango_limite)[0][-1] + 1
        X_all = X_all[:, :idx_max]
    
    return X_all, y_all

## SVR

In [3]:
import os
import numpy as np
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

print("="*60)
print("Evaluación de SVR con validación cruzada (5 folds)")
print("="*60)

for pretratamiento in ['orig', 'sg', 'msc', 'snv']:
    for nombre_rango, limite in rangos.items():
        X_all, y_all = get_data_by_range(data_dict, pretratamiento, limite)

        params = best_hyperparams['svr'][pretratamiento]['best_params']

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(
                C=params['C'],
                epsilon=params['epsilon'],
                kernel=params['kernel'],
                gamma=params['gamma']
            ))
        ])

        scoring = {
            'rmse': 'neg_root_mean_squared_error',
            'r2': 'r2',
            'mae': 'neg_mean_absolute_error'
        }

        scores = cross_validate(model, X_all, y_all, cv=kf, scoring=scoring, return_train_score=False)

        rmse_list = -scores['test_rmse']
        mae_list = -scores['test_mae']
        r2_list = scores['test_r2']

        rpd_list = []
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_all)):
            y_test_fold = y_all[test_idx]
            std_y = np.std(y_test_fold, ddof=1)
            rpd_list.append(std_y / (rmse_list[fold_idx] + 1e-12))

        # Predicciones out-of-fold
        y_pred_oof = cross_val_predict(model, X_all, y_all, cv=kf)

        key = f"{pretratamiento}_{nombre_rango}"
        results_cv['svr'][key] = {
            'rmse': rmse_list.tolist(),
            'mae': mae_list.tolist(),
            'r2': r2_list.tolist(),
            'rpd': rpd_list,
            'y_true_oof': y_all.tolist(),
            'y_pred_oof': y_pred_oof.tolist()
        }

        print(f"\nSVR | {pretratamiento} | {nombre_rango}")
        print(f"  RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}")
        print(f"  R²:   {np.mean(r2_list):.4f} ± {np.std(r2_list):.4f}")
        print(f"  MAE:  {np.mean(mae_list):.4f} ± {np.std(mae_list):.4f}")
        print(f"  RPD:  {np.mean(rpd_list):.2f} ± {np.std(rpd_list):.2f}")

Evaluación de SVR con validación cruzada (5 folds)

SVR | orig | completo
  RMSE: 4.3977 ± 0.1080
  R²:   0.9521 ± 0.0024
  MAE:  3.2603 ± 0.0831
  RPD:  4.58 ± 0.12

SVR | orig | hasta_1700
  RMSE: 4.8392 ± 0.1541
  R²:   0.9421 ± 0.0029
  MAE:  3.2849 ± 0.0419
  RPD:  4.16 ± 0.10

SVR | orig | hasta_1400
  RMSE: 6.7668 ± 0.1599
  R²:   0.8868 ± 0.0032
  MAE:  4.6797 ± 0.0734
  RPD:  2.97 ± 0.04

SVR | sg | completo
  RMSE: 3.7908 ± 0.0859
  R²:   0.9645 ± 0.0010
  MAE:  2.7174 ± 0.0297
  RPD:  5.31 ± 0.07

SVR | sg | hasta_1700
  RMSE: 4.2563 ± 0.1297
  R²:   0.9552 ± 0.0019
  MAE:  2.8770 ± 0.0471
  RPD:  4.73 ± 0.10

SVR | sg | hasta_1400
  RMSE: 6.2072 ± 0.1533
  R²:   0.9047 ± 0.0030
  MAE:  4.1586 ± 0.0503
  RPD:  3.24 ± 0.05

SVR | msc | completo
  RMSE: 4.4358 ± 0.1187
  R²:   0.9514 ± 0.0010
  MAE:  3.3616 ± 0.0952
  RPD:  4.54 ± 0.05

SVR | msc | hasta_1700
  RMSE: 4.5572 ± 0.0721
  R²:   0.9486 ± 0.0019
  MAE:  3.3590 ± 0.0459
  RPD:  4.42 ± 0.08

SVR | msc | hasta_1400
  R

## Random Forest

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, cross_val_predict

print("\n" + "="*60)
print("Evaluación de Random Forest con validación cruzada (5 folds)")
print("="*60)

for pretratamiento in ['orig', 'sg', 'msc', 'snv']:
    for nombre_rango, limite in rangos.items():
        X_all, y_all = get_data_by_range(data_dict, pretratamiento, limite)

        params = best_hyperparams['random_forest'][pretratamiento]['best_params']

        model = RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            min_samples_leaf=params['min_samples_leaf'],
            max_features=params['max_features'],
            random_state=42,
            n_jobs=-1
        )

        scoring = {
            'rmse': 'neg_root_mean_squared_error',
            'r2': 'r2',
            'mae': 'neg_mean_absolute_error'
        }

        scores = cross_validate(model, X_all, y_all, cv=kf, scoring=scoring, return_train_score=False)

        rmse_list = -scores['test_rmse']
        mae_list = -scores['test_mae']
        r2_list = scores['test_r2']

        rpd_list = []
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_all)):
            y_test_fold = y_all[test_idx]
            std_y = np.std(y_test_fold, ddof=1)
            rpd_list.append(std_y / (rmse_list[fold_idx] + 1e-12))

        # Predicciones out-of-fold
        y_pred_oof = cross_val_predict(model, X_all, y_all, cv=kf)

        key = f"{pretratamiento}_{nombre_rango}"
        results_cv['random_forest'][key] = {
            'rmse': rmse_list.tolist(),
            'mae': mae_list.tolist(),
            'r2': r2_list.tolist(),
            'rpd': rpd_list,
            'y_true_oof': y_all.tolist(),
            'y_pred_oof': y_pred_oof.tolist()
        }

        print(f"\nRF | {pretratamiento} | {nombre_rango}")
        print(f"  RMSE: {np.mean(rmse_list):.4f} ± {np.std(rmse_list):.4f}")
        print(f"  R²:   {np.mean(r2_list):.4f} ± {np.std(r2_list):.4f}")
        print(f"  MAE:  {np.mean(mae_list):.4f} ± {np.std(mae_list):.4f}")
        print(f"  RPD:  {np.mean(rpd_list):.2f} ± {np.std(rpd_list):.2f}")


Evaluación de Random Forest con validación cruzada (5 folds)

RF | orig | completo
  RMSE: 3.9523 ± 0.1658
  R²:   0.9614 ± 0.0021
  MAE:  2.7689 ± 0.0752
  RPD:  5.10 ± 0.14

RF | orig | hasta_1700
  RMSE: 3.5165 ± 0.1462
  R²:   0.9694 ± 0.0022
  MAE:  2.3524 ± 0.0532
  RPD:  5.73 ± 0.20

RF | orig | hasta_1400
  RMSE: 2.9081 ± 0.1419
  R²:   0.9790 ± 0.0023
  MAE:  1.7608 ± 0.0448
  RPD:  6.94 ± 0.37

RF | sg | completo
  RMSE: 3.3844 ± 0.1830
  R²:   0.9717 ± 0.0023
  MAE:  2.2381 ± 0.0889
  RPD:  5.96 ± 0.24

RF | sg | hasta_1700
  RMSE: 2.9470 ± 0.1749
  R²:   0.9785 ± 0.0024
  MAE:  1.8318 ± 0.0514
  RPD:  6.85 ± 0.36

RF | sg | hasta_1400
  RMSE: 2.4247 ± 0.1806
  R²:   0.9853 ± 0.0024
  MAE:  1.3340 ± 0.0505
  RPD:  8.35 ± 0.66

RF | msc | completo
  RMSE: 4.2384 ± 0.1199
  R²:   0.9556 ± 0.0011
  MAE:  3.0356 ± 0.0484
  RPD:  4.75 ± 0.06

RF | msc | hasta_1700
  RMSE: 3.9260 ± 0.1164
  R²:   0.9619 ± 0.0006
  MAE:  2.7739 ± 0.0456
  RPD:  5.13 ± 0.04

RF | msc | hasta_1400
 

## CNN

In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# Asegurar que la clase NIR_CNN esté definida (la misma que usaste antes)
class NIR_CNN(nn.Module):
    def __init__(self, n_features, n_filters=32, kernel_size=5, n_conv_blocks=2, hidden_dim=128, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = 1
        for i in range(n_conv_blocks):
            out_ch = n_filters * (2**i if i>0 else 1)
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.MaxPool1d(kernel_size=2))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch

        self.conv = nn.Sequential(*layers)

        # calcular tamaño después de las maxpool (cada bloque divide por 2)
        conv_out_len = n_features // (2**n_conv_blocks)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * conv_out_len, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)            # -> (batch, 1, n_features)
        x = self.conv(x)              # -> (batch, channels, len)
        x = self.fc(x)
        return x.squeeze(1)


def evaluate_cnn_fold(X_train_fold, y_train_fold, X_test_fold, y_test_fold, params, device, max_epochs=40, patience=6):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_fold)
    y_train_scaled = scaler_y.fit_transform(y_train_fold.reshape(-1, 1)).ravel()
    X_test_scaled = scaler_X.transform(X_test_fold)

    train_dataset_full = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32)
    )

    val_size = int(len(train_dataset_full) * 0.15)
    train_size = len(train_dataset_full) - val_size
    train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False)

    model = NIR_CNN(
        n_features=X_train_fold.shape[1],
        n_filters=params['n_filters'],
        kernel_size=params['kernel_size'],
        n_conv_blocks=params['n_conv_blocks'],
        hidden_dim=params['hidden'],
        dropout=params['dropout']
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(Xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                outputs = model(Xb)
                loss = criterion(outputs, yb)
                val_loss += loss.item() * Xb.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    model.eval()
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        y_pred_scaled = model(X_test_t).cpu().numpy().ravel()

    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

    rmse = sqrt(mean_squared_error(y_test_fold, y_pred))
    mae = mean_absolute_error(y_test_fold, y_pred)
    r2 = r2_score(y_test_fold, y_pred)
    std_y = np.std(y_test_fold, ddof=1)
    rpd = std_y / (rmse + 1e-12)

    return rmse, mae, r2, rpd, y_pred


print("\n" + "="*60)
print("Evaluación de CNN con validación cruzada (5 folds)")
print("="*60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for pretratamiento in ['orig', 'sg', 'msc', 'snv']:
    for nombre_rango, limite in rangos.items():
        X_all, y_all = get_data_by_range(data_dict, pretratamiento, limite)

        if pretratamiento not in best_hyperparams['cnn']:
            print(f"  No hay hiperparámetros para CNN en {pretratamiento}. Saltando.")
            continue

        params = best_hyperparams['cnn'][pretratamiento]['best_params']

        rmse_folds, mae_folds, r2_folds, rpd_folds = [], [], [], []

        y_true_oof = np.zeros(len(y_all))
        y_pred_oof = np.zeros(len(y_all))

        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_all)):
            X_train_fold = X_all[train_idx]
            y_train_fold = y_all[train_idx]
            X_test_fold = X_all[test_idx]
            y_test_fold = y_all[test_idx]

            rmse, mae, r2, rpd, y_pred = evaluate_cnn_fold(
                X_train_fold, y_train_fold,
                X_test_fold, y_test_fold,
                params, device,
                max_epochs=40,
                patience=6
            )

            rmse_folds.append(rmse)
            mae_folds.append(mae)
            r2_folds.append(r2)
            rpd_folds.append(rpd)

            y_true_oof[test_idx] = y_test_fold
            y_pred_oof[test_idx] = y_pred

        key = f"{pretratamiento}_{nombre_rango}"
        results_cv['cnn'][key] = {
            'rmse': rmse_folds,
            'mae': mae_folds,
            'r2': r2_folds,
            'rpd': rpd_folds,
            'y_true_oof': y_true_oof.tolist(),
            'y_pred_oof': y_pred_oof.tolist()
        }

        print(f"\nCNN | {pretratamiento} | {nombre_rango}")
        print(f"  RMSE: {np.mean(rmse_folds):.4f} ± {np.std(rmse_folds):.4f}")
        print(f"  R²:   {np.mean(r2_folds):.4f} ± {np.std(r2_folds):.4f}")
        print(f"  MAE:  {np.mean(mae_folds):.4f} ± {np.std(mae_folds):.4f}")
        print(f"  RPD:  {np.mean(rpd_folds):.2f} ± {np.std(rpd_folds):.2f}")


Evaluación de CNN con validación cruzada (5 folds)
Usando dispositivo: cpu


CNN | orig | completo
  RMSE: 5.0177 ± 0.3346
  R²:   0.9374 ± 0.0084
  MAE:  3.7939 ± 0.2919
  RPD:  4.03 ± 0.27

CNN | orig | hasta_1700
  RMSE: 4.5429 ± 0.2180
  R²:   0.9490 ± 0.0027
  MAE:  3.4271 ± 0.1556
  RPD:  4.43 ± 0.12

CNN | orig | hasta_1400
  RMSE: 3.5980 ± 0.4364
  R²:   0.9678 ± 0.0070
  MAE:  2.7325 ± 0.3511
  RPD:  5.65 ± 0.53

CNN | sg | completo
  RMSE: 4.4972 ± 0.2241
  R²:   0.9500 ± 0.0030
  MAE:  3.3822 ± 0.1120
  RPD:  4.48 ± 0.13

CNN | sg | hasta_1700
  RMSE: 3.8500 ± 0.2406
  R²:   0.9632 ± 0.0041
  MAE:  2.8885 ± 0.1877
  RPD:  5.24 ± 0.32

CNN | sg | hasta_1400
  RMSE: 3.2180 ± 0.1657
  R²:   0.9743 ± 0.0025
  MAE:  2.4337 ± 0.1460
  RPD:  6.27 ± 0.31

CNN | msc | completo
  RMSE: 3.9545 ± 0.2911
  R²:   0.9609 ± 0.0070
  MAE:  2.9550 ± 0.1764
  RPD:  5.12 ± 0.46

CNN | msc | hasta_1700
  RMSE: 3.6582 ± 0.2103
  R²:   0.9667 ± 0.0046
  MAE:  2.7714 ± 0.1690
  RPD:  5.52 ± 0.38



## Guardar todos los resultados para ANOVA

In [15]:
# Guardar los resultados en un archivo para análisis posterior
import pickle
with open("results_cv_all.pkl", "wb") as f:
    pickle.dump(results_cv, f)

print("\nResultados guardados en results_cv_all.pkl")


Resultados guardados en results_cv_all.pkl
